# Laboratorio simbólico: coordenadas y cambio de base

Trabajaremos con `sympy` para conservar fracciones y parámetros exactos. En
cada sección se verificará que dos recorridos de coordenadas producen el mismo
vector.


In [ ]:
import sympy as sp
from sympy import Matrix
sp.init_printing()

def verificar_base(B):
    return B.rows == B.cols and B.rank() == B.cols

def coords_en_base(x, B):
    # Resuelve B c = x sin formar explícitamente B^{-1}.
    if not verificar_base(B):
        raise ValueError("Las columnas de B no forman una base del espacio.")
    return B.LUsolve(x)

def cambio_base(B, C):
    # P_{B->C}: recibe coordenadas B y entrega coordenadas C.
    if not verificar_base(B) or not verificar_base(C) or B.rows != C.rows:
        raise ValueError("B y C deben ser matrices de bases del mismo espacio.")
    return C.inv() * B

def matriz_transformacion_en_bases(A, B_dom, C_cod):
    # Si A usa bases canónicas, devuelve [T]_{C <- B}.
    if A.cols != B_dom.rows or A.rows != C_cod.rows:
        raise ValueError("Dimensiones incompatibles.")
    return C_cod.inv() * A * B_dom


## 1. El vector y sus coordenadas

Para $\mathcal B=((1,0),(1,1))$ y $x=(3,5)$ resolvemos
$M_{\mathcal B}[x]_{\mathcal B}=x$.


In [ ]:
B = Matrix([[1, 1], [0, 1]])
x = Matrix([3, 5])
x_B = coords_en_base(x, B)

display(x_B)
display(B * x_B)
assert B * x_B == x


## 2. Cambio entre dos bases de $\mathbb R^3$

Comparamos el cálculo directo $M_{\mathcal C}^{-1}x$ con el recorrido
$[x]_{\mathcal B}\mapsto[x]_{\mathcal C}$.


In [ ]:
B = sp.eye(3)
C = Matrix([[1, 1, 0],
            [1, -1, 0],
            [0, 0, 1]])
x = Matrix([1, 2, 3])

P_BC = cambio_base(B, C)
P_CB = cambio_base(C, B)
x_B = coords_en_base(x, B)
x_C_directo = coords_en_base(x, C)
x_C_por_cambio = P_BC * x_B

display(P_BC, P_CB, x_C_directo)
assert x_C_directo == x_C_por_cambio
assert P_CB * P_BC == sp.eye(3)


## 3. Base con parámetro simbólico

La matriz $M_{\mathcal B}=\begin{bmatrix}1&0\\a&1\end{bmatrix}$ tiene
determinante $1$ para todo $a\in\mathbb R$.


In [ ]:
a, x1, x2 = sp.symbols('a x1 x2')
B = Matrix([[1, 0], [a, 1]])
x = Matrix([x1, x2])
x_B = coords_en_base(x, B)

display(x_B)
assert sp.simplify(B * x_B - x) == sp.zeros(2, 1)


## 4. Matriz de una transformación en bases elegidas

Sea $A$ la matriz canónica de $T:\mathbb R^2\to\mathbb R^2$. Verificamos

$$[T]_{\mathcal C\leftarrow\mathcal B}=M_{\mathcal C}^{-1}AM_{\mathcal B}.$$


In [ ]:
A = Matrix([[2, 1], [0, 3]])
B = Matrix([[1, 1], [0, 1]])
C = Matrix([[1, 0], [1, 1]])
T_CB = matriz_transformacion_en_bases(A, B, C)

u, v = sp.symbols('u v')
x_B = Matrix([u, v])
via_canonica = C.inv() * A * B * x_B
via_relativa = T_CB * x_B

display(T_CB, via_relativa)
assert sp.simplify(via_canonica - via_relativa) == sp.zeros(2, 1)


## 5. Dominio y codominio de dimensiones distintas

Para $T:\mathbb R^3\to\mathbb R^2$, la matriz relativa tiene tamaño $2\times3$.


In [ ]:
A = Matrix([[1, 2, 0], [0, 1, 1]])
B = Matrix([[1, 1, 0], [0, 1, 1], [0, 0, 1]])
C = Matrix([[1, 1], [1, 0]])
T_CB = matriz_transformacion_en_bases(A, B, C)

alpha, beta, gamma = sp.symbols('alpha beta gamma')
x_B = Matrix([alpha, beta, gamma])
assert T_CB.shape == (2, 3)
assert sp.simplify(C.inv() * A * B * x_B - T_CB * x_B) == sp.zeros(2, 1)
display(T_CB)


## 6. Cambio simultáneo de bases

Partimos de $[T]_{\mathcal C\leftarrow\mathcal B}$ y volvemos a las bases
canónicas. Los factores se colocan según las coordenadas que reciben y entregan.


In [ ]:
B_nueva = sp.eye(3)
C_nueva = sp.eye(2)

P_Bnueva_B = cambio_base(B_nueva, B)
P_C_Cnueva = cambio_base(C, C_nueva)
T_nueva = P_C_Cnueva * T_CB * P_Bnueva_B

display(T_nueva)
assert sp.simplify(T_nueva - A) == sp.zeros(2, 3)


## 7. Caso especial: base ortonormal

Para una rotación, las columnas de $Q$ son ortonormales y $Q^{-1}=Q^T$.


In [ ]:
theta = sp.symbols('theta', real=True)
Q = Matrix([[sp.cos(theta), -sp.sin(theta)],
            [sp.sin(theta),  sp.cos(theta)]])

display(sp.simplify(Q.T * Q))
assert sp.simplify(Q.T * Q - sp.eye(2)) == sp.zeros(2)
assert sp.simplify(Q.inv() - Q.T) == sp.zeros(2)


## 8. Plantilla validada

La función rechaza matrices singulares antes de llamarlas bases. Prueba con tus
propios datos cambiando $B$, $C$ y $x$.


In [ ]:
B = Matrix([[1, 0, 1], [0, 1, 1], [0, 0, 1]])
C = Matrix([[1, 1, 0], [0, 1, 1], [1, 0, 0]])
x = Matrix([1, 2, 3])

assert verificar_base(B) and verificar_base(C)
P_BC = cambio_base(B, C)
x_B = coords_en_base(x, B)
x_C = coords_en_base(x, C)

display(P_BC, x_B, x_C)
assert P_BC * x_B == x_C


## 9. Para explorar

1. Construye tres bases y verifica la ley de composición de matrices de cambio.
2. Calcula una matriz relativa formando sus columnas $[T(b_j)]_{\mathcal C}$ y
   compárala con la fórmula matricial.
3. Sustituye $C$ en la plantilla por una matriz singular. Interpreta el error.
4. Elige dos bases distintas para un operador y verifica la relación de
   semejanza entre sus matrices.
